In [1]:
"""Train a baseline gradient-boosting model per horizon and write predictions.parquet.

Baseline approach: histogram GBM regression (sklearn's LightGBM-equivalent)
on each target (target_10d, target_30d), with a time-based holdout for a
sanity-check Spearman score. Predictions are rank-normalized to [0, 1]
per the submission spec (id, pred_10d, pred_30d).
"""

from pathlib import Path

import polars as pl
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression, Lasso

DATA_DIR = Path("data") # folder
OUT_DIR = Path("predictions") # folder
OUT_DIR.mkdir(exist_ok=True)

train = pl.read_parquet(DATA_DIR / "training_data.parquet")
infer = pl.read_parquet(DATA_DIR / "inference_data.parquet")

feature_cols = [c for c in train.columns if c.startswith("feature_")] # extract the feature columns
print(f"{len(feature_cols)} features, {train.height:,} training rows, {infer.height} inference rows")

180 features, 195,021 training rows, 169 inference rows


In [2]:
def ts_split(train, lookback):
    # time-based holdout: last 60 dates for validation
    dates = train["date"].unique().sort()
    split_date = dates[-lookback] # validation holdout
    tr = train.filter(pl.col("date") < split_date) # filter up to last training_date
    va = train.filter(pl.col("date") >= split_date) # validation date and beyond
    return tr, va

In [3]:
def Ridge_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = Ridge(alpha=1.0, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['Ridge'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['Ridge'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [4]:
def ElasticNet_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        # alpha is much smaller than Ridge's: ElasticNet's L1 part zeroes out
        # coefficients aggressively, and at alpha=1.0 it would kill all 180
        model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['ElasticNet'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['ElasticNet'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [5]:
def Linear_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = LinearRegression() # plain OLS - no regularization, no knobs
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['LR'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['LR'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [12]:
def Lasso_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = Lasso(alpha=0.0001) # plain OLS - no regularization, no knobs
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['LASSO'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['LASSO'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [13]:
lookback = 60
MODELS = ["Ridge", "ElasticNet", "LR", "LASSO"] # one key per model
preds = {name: {"id": infer["id"]} for name in MODELS} # predictions

tr, va = ts_split(train, lookback)
print("-- Ridge --")
preds = Ridge_regression(tr, va, preds)
print("-- ElasticNet --")
preds = ElasticNet_regression(tr, va, preds)
print("-- LinearRegression --")
preds = Linear_regression(tr, va, preds)
print("--LASSO--")
final = Lasso_regression(tr, va, preds)

-- Ridge --
target_10d: holdout Spearman = 0.0884
target_30d: holdout Spearman = 0.1085
-- ElasticNet --
target_10d: holdout Spearman = 0.1046
target_30d: holdout Spearman = 0.1677
-- LinearRegression --
target_10d: holdout Spearman = 0.0882
target_30d: holdout Spearman = 0.1082
--LASSO--
target_10d: holdout Spearman = 0.1050


/Users/roofernando/Cloud/algoChains/github/crowdcent-hyperliquid/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.601681e+00, tolerance: 1.498e+00
  model = cd_fast.enet_coordinate_descent(


target_30d: holdout Spearman = 0.1293


-- Ridge --
target_10d: holdout Spearman = 0.0893
target_30d: holdout Spearman = 0.0989
-- ElasticNet --
target_10d: holdout Spearman = 0.0992
target_30d: holdout Spearman = 0.1562
-- LinearRegression --
target_10d: holdout Spearman = 0.0890
target_30d: holdout Spearman = 0.0986
-- HGBR --
target_10d: holdout Spearman = 0.0762
target_30d: holdout Spearman = 0.1708

In [8]:
# Convert the predictions for display/inspection (better structure)
# one column per model+horizon, joined on 'id' - ready for aggregation later
df = None
for name in MODELS:
    df_m = pl.DataFrame({
        "id": final[name]["id"],
        f"{name}_pred_10d": final[name]["pred_10d"],
        f"{name}_pred_30d": final[name]["pred_30d"],
    })
    df = df_m if df is None else df.join(df_m, on="id", how="inner")

df.head()

id,Ridge_pred_10d,Ridge_pred_30d,ElasticNet_pred_10d,ElasticNet_pred_30d,LR_pred_10d,LR_pred_30d,LASSO_pred_10d,LASSO_pred_30d
str,f64,f64,f64,f64,f64,f64,f64,f64
"""0G""",0.378698,0.248521,0.16568,0.136095,0.372781,0.236686,0.502959,0.502959
"""2Z""",0.449704,0.278107,0.08284,0.065089,0.449704,0.284024,0.502959,0.502959
"""AAVE""",0.733728,0.692308,0.662722,0.775148,0.727811,0.692308,0.502959,0.502959
"""ACE""",0.142012,0.213018,0.213018,0.260355,0.147929,0.224852,0.502959,0.502959
"""ADA""",0.887574,0.704142,0.751479,0.816568,0.887574,0.704142,0.502959,0.502959


In [ ]:
def spearman_weights(final, va, target):
    # per-model weight = its validation-holdout Spearman, floored at 0 so a
    # model with negative correlation gets no vote, then normalized to sum to 1
    horizon = target.replace("target", "pred")
    va_t = va.drop_nulls(subset=[target]) # same rows every model predicted on
    y = va_t[target].to_numpy()
    scores = {name: max(spearmanr(y, final[name][f"val_{horizon}"].to_numpy())[0], 0.0) for name in MODELS}
    total = sum(scores.values())
    if total == 0: # every model scored <= 0 on the holdout: fall back to equal weights
        return {name: 1 / len(MODELS) for name in scores}
    return {name: s / total for name, s in scores.items()}

def aggregate(final, va):
    # performance-weighted ensemble: weight each model's rank column by its
    # validation Spearman (weights sum to 1), then re-rank the weighted
    # average back to (0, 1] so the output is a valid submission
    agg = {"id": final[MODELS[0]]["id"]} # ids are identical across models
    for target in ["target_10d", "target_30d"]:
        horizon = target.replace("target", "pred")
        weights = spearman_weights(final, va, target)
        print(f"{horizon} weights: " + ", ".join(f"{n}={w:.3f}" for n, w in weights.items()))
        stacked = pl.DataFrame({name: final[name][horizon] * weights[name] for name in MODELS}) # one weighted column per model
        weighted_rank = stacked.sum_horizontal() # weighted average (weights already sum to 1)
        agg[horizon] = weighted_rank.rank() / len(weighted_rank) # re-rank to (0, 1]
    return pl.DataFrame(agg) # submission format: id, pred_10d, pred_30d


In [ ]:
# ensemble sanity check: same performance-weighted aggregation as aggregate(),
# but on the validation holdout, so we can Spearman the aggregate against the
# actual targets. Caveat: the weights are fit on this same holdout, so this
# number runs slightly optimistic.
for target in ["target_10d", "target_30d"]:
    horizon = target.replace("target", "pred")
    va_t = va.drop_nulls(subset=[target]) # same rows every model predicted on
    weights = spearman_weights(final, va, target)
    stacked = pl.DataFrame({name: final[name][f"val_{horizon}"] * weights[name] for name in MODELS}) # one weighted column per model
    weighted_rank = stacked.sum_horizontal() # weighted average (weights already sum to 1)
    corr, _ = spearmanr(va_t[target].to_numpy(), weighted_rank.to_numpy())
    print(f"{target}: ensemble holdout Spearman = {corr:.4f}")


In [11]:
submission = aggregate(final, va) # final targets: aggregate pred_10d and pred_30d

# sanity checks against the submission spec before writing
assert submission.columns == ["id", "pred_10d", "pred_30d"] # exact required columns
assert submission["pred_10d"].is_between(0, 1).all() # floats in [0, 1]
assert submission["pred_30d"].is_between(0, 1).all()
assert submission.height >= 80 # minimum 80 assets

submission.write_parquet(OUT_DIR / "model_slot3.parquet") # wrap it up in a parquet
print(f"wrote {OUT_DIR / 'model_slot3.parquet'} ({submission.height} assets)")
submission.head()

wrote predictions/model_slot3.parquet (169 assets)


id,pred_10d,pred_30d
str,f64,f64
"""0G""",0.295858,0.207101
"""2Z""",0.319527,0.213018
"""AAVE""",0.721893,0.727811
"""ACE""",0.142012,0.230769
"""ADA""",0.843195,0.745562
